In [1]:
import os
import time
import random
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Configuration
TRAITS = ["cOPN", "cCON", "cEXT", "cAGR",  "cNEU"]
TRAIT_NAMES = {
    "cOPN": "Openness",
    "cCON": "Conscientiousness",
    "cEXT": "Extraversion",
    "cAGR": "Agreeableness",
    "cNEU": "Neuroticism",
}

# Paths
DATA_DIR = "../../data/split/essays"
EXP_DIR  = "../../experiment/LSTM"
MODEL_DIR = "../../model/LSTM"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(EXP_DIR, exist_ok=True)

# Hyperparameters
MAX_LEN        = 512
EMBED_DIM      = 128
HIDDEN_DIM     = 128
NUM_LAYERS     = 2
DROPOUT        = 0.3
BATCH_SIZE     = 32
EPOCHS         = 50
LEARNING_RATE  = 0.001
EARLY_STOP_PATIENCE = 4

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Experiment directory: {EXP_DIR}")

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

Device: cpu
Experiment directory: ../../experiment/LSTM


In [3]:
#Data loading and exploration
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
val_df   = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"Train: {train_df.shape[0]} samples | Val: {val_df.shape[0]} samples | Test: {test_df.shape[0]} samples")
print(f"Columns: {list(train_df.columns)}")

# Class distribution per trait
print("\nClass distribution (train set):")
for trait in TRAITS:
    print(f"  {trait} ({TRAIT_NAMES[trait]}): "
          f"high={sum(train_df[trait]=='high')}, "
          f"low={sum(train_df[trait]=='low')}")

# Text length statistics
train_df['text_len'] = train_df['text'].apply(len)
print(f"\nText length stats (characters):")
print(f"  Mean: {train_df['text_len'].mean():.0f}")
print(f"  Median: {train_df['text_len'].median():.0f}")
print(f"  Max: {train_df['text_len'].max():.0f}")


Train: 1974 samples | Val: 247 samples | Test: 247 samples
Columns: ['text', 'cEXT', 'cNEU', 'cAGR', 'cCON', 'cOPN', 'label']

Class distribution (train set):
  cOPN (Openness): high=1017, low=957
  cCON (Conscientiousness): high=1004, low=970
  cEXT (Extraversion): high=1022, low=952
  cAGR (Agreeableness): high=1046, low=928
  cNEU (Neuroticism): high=986, low=988

Text length stats (characters):
  Mean: 3306
  Median: 3180
  Max: 12852


In [4]:
# Text preprocessing and tokenization
import re
import collections

def clean_text(text):
    """Lowercase and keep alphanumeric + spaces for word tokenization."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text, max_len=MAX_LEN):
    """Simple word-level tokenization with fixed max length."""
    words = clean_text(text).split()
    if len(words) > max_len:
        words = words[:max_len]
    return words

# Build vocabulary from training set
all_words = []
for text in train_df['text']:
    all_words.extend(tokenize(text))

word_counts = collections.Counter(all_words)
print(f"Total word tokens in train: {len(all_words)}")
print(f"Unique words (raw): {len(word_counts)}")

# Build vocabulary: map word -> index (0 = pad, 1 = unk)
MIN_FREQ = 2
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, count in word_counts.items():
    if count >= MIN_FREQ:
        vocab[word] = len(vocab)

print(f"Vocabulary size (min_freq={MIN_FREQ}): {len(vocab)}")

def encode(text, vocab, max_len=MAX_LEN):
    """Encode text to word indices."""
    tokens = tokenize(text, max_len)
    encoded = [vocab.get(w, vocab['<UNK>']) for w in tokens]
    # Pad
    if len(encoded) < max_len:
        encoded += [vocab['<PAD>']] * (max_len - len(encoded))
    return encoded

# Encode all splits
X_train = np.array([encode(t, vocab) for t in train_df['text']])
X_val   = np.array([encode(t, vocab) for t in val_df['text']])
X_test  = np.array([encode(t, vocab) for t in test_df['text']])

print(f"\nX_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")


Total word tokens in train: 938562
Unique words (raw): 22083
Vocabulary size (min_freq=2): 11315

X_train shape: (1974, 512)
X_val shape:   (247, 512)
X_test shape:  (247, 512)


In [5]:
# Dataset class and dataloaders
class EssayDataset(Dataset):
    def __init__(self, texts, labels_dict):
        """
        texts: numpy array of encoded word indices (N x MAX_LEN)
        labels_dict: {trait: numpy array of 0/1 labels}
        """
        self.texts = torch.LongTensor(texts)
        self.labels = {k: torch.FloatTensor(v) for k, v in labels_dict.items()}
        self.n_samples = len(texts)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        x = self.texts[idx]
        ys = {k: v[idx] for k, v in self.labels.items()}
        return x, ys

# Convert labels: high=1, low=0
y_train = {trait: (train_df[trait] == 'high').astype(int).values for trait in TRAITS}
y_val   = {trait: (val_df[trait] == 'high').astype(int).values   for trait in TRAITS}
y_test  = {trait: (test_df[trait] == 'high').astype(int).values  for trait in TRAITS}

train_dataset = EssayDataset(X_train, y_train)
val_dataset   = EssayDataset(X_val,   y_val)
test_dataset  = EssayDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")


Train batches: 62 | Val batches: 8 | Test batches: 8


In [6]:
# LSTM model definition
class PersonalityLSTM(nn.Module):
    """
    Bidirectional LSTM with shared text encoder and per-trait classification heads.
    Embedding -> BiLSTM -> [CLS-like] mean pooling -> FC -> trait outputs.
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers,
                 dropout, num_traits=5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        # 2 * hidden_dim for bidirectional
        self.fc = nn.Linear(hidden_dim * 2, 64)
        self.relu = nn.ReLU()
        # Separate head per trait
        self.heads = nn.ModuleDict({
            trait: nn.Linear(64, 1) for trait in TRAITS
        })

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embedding(x)           # (batch, seq_len, embed_dim)
        lstm_out, _ = self.lstm(emb)       # (batch, seq_len, hidden_dim*2)

        # Mean pooling over sequence (ignoring padding)
        mask = (x != 0).unsqueeze(-1).float()
        pooled = (lstm_out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        feat = self.dropout(pooled)
        feat = self.relu(self.fc(feat))

        outputs = {trait: torch.sigmoid(self.heads[trait](feat)).squeeze(-1) for trait in TRAITS}
        return outputs

# Instantiate model
VOCAB_SIZE = len(vocab)
model = PersonalityLSTM(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: PersonalityLSTM")
print(f"  Vocabulary size: {VOCAB_SIZE}")
print(f"  Embed dim: {EMBED_DIM} | Hidden dim: {HIDDEN_DIM} | Num layers: {NUM_LAYERS}")
print(f"  Total params: {total_params:,} | Trainable: {trainable_params:,}")

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


Model: PersonalityLSTM
  Vocabulary size: 11315
  Embed dim: 128 | Hidden dim: 128 | Num layers: 2
  Total params: 2,124,549 | Trainable: 2,124,549


In [7]:
# Training loop with per-trait early stopping
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch_idx, (x, ys_true) in enumerate(loader):
        x = x.to(device)
        ys_true = {k: v.to(device) for k, v in ys_true.items()}

        optimizer.zero_grad()
        ys_pred = model(x)

        loss = sum(criterion(ys_pred[t], ys_true[t]) for t in TRAITS) / len(TRAITS)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds  = {t: [] for t in TRAITS}
    all_labels = {t: [] for t in TRAITS}

    with torch.no_grad():
        for x, ys_true in loader:
            x = x.to(device)
            ys_true = {k: v.to(device) for k, v in ys_true.items()}
            ys_pred = model(x)

            loss = sum(criterion(ys_pred[t], ys_true[t]) for t in TRAITS) / len(TRAITS)
            total_loss += loss.item()

            for t in TRAITS:
                all_preds[t].extend((ys_pred[t].cpu() >= 0.5).long().numpy().tolist())
                all_labels[t].extend(ys_true[t].cpu().long().numpy().tolist())

    avg_loss = total_loss / len(loader)
    return avg_loss, all_preds, all_labels

# Training
import json
run_id = time.strftime('%Y%m%d-%H%M%S')
best_model_path = os.path.join(MODEL_DIR, f'best_model_{run_id}.pt')

history = {t: {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []} for t in TRAITS}
history['epoch'] = []
best_val_f1 = 0.0
patience_counter = 0

print(f"\n{'='*70}")
print(f"Training LSTM — Run ID: {run_id}")
print(f"{'='*70}")
print(f"{'Epoch':>5} | {'Train Loss':>11} | {'Val Loss':>9} | {'Val F1 per trait':<45}")
print(f"{'-'*70}")

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion, device)

    val_acc = {}
    val_f1  = {}
    for t in TRAITS:
        acc = np.mean(np.array(val_preds[t]) == np.array(val_labels[t]))
        f1  = f1_score(np.array(val_labels[t]), np.array(val_preds[t]), average="weighted", zero_division=0)
        val_acc[t] = acc
        val_f1[t]  = f1

    avg_val_f1 = np.mean(list(val_f1.values()))

    print(f"{epoch:>5} | {train_loss:>11.4f} | {val_loss:>9.4f} | "
          f"EXT={val_f1['cEXT']:.3f} NEU={val_f1['cNEU']:.3f} "
          f"AGR={val_f1['cAGR']:.3f} CON={val_f1['cCON']:.3f} OPN={val_f1['cOPN']:.3f} | AvgF1={avg_val_f1:.3f}")

    history['epoch'].append(epoch)
    for t in TRAITS:
        history[t]['train_loss'].append(train_loss)
        history[t]['val_loss'].append(val_loss)
        history[t]['val_acc'].append(val_acc[t])
        history[t]['val_f1'].append(val_f1[t])

    # Save best model based on average validation F1
    if avg_val_f1 > best_val_f1:
        best_val_f1 = avg_val_f1
        torch.save(model.state_dict(), best_model_path)
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"{'='*70}")
print(f"Best validation F1: {best_val_f1:.4f}")
print(f"Model saved to: {best_model_path}")

# Load best model for evaluation
model.load_state_dict(torch.load(best_model_path, weights_only=True))

# Evaluate on test set
test_loss, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
print(f"\nTest Loss: {test_loss:.4f}")


Training LSTM — Run ID: 20260325-173143
Epoch |  Train Loss |  Val Loss | Val F1 per trait                             
----------------------------------------------------------------------
    1 |      0.6933 |    0.6919 | EXT=0.549 NEU=0.331 AGR=0.368 CON=0.427 OPN=0.354 | AvgF1=0.406
    2 |      0.6881 |    0.6865 | EXT=0.552 NEU=0.338 AGR=0.494 CON=0.581 OPN=0.576 | AvgF1=0.508
    3 |      0.6791 |    0.6848 | EXT=0.567 NEU=0.524 AGR=0.490 CON=0.592 OPN=0.601 | AvgF1=0.555
    4 |      0.6724 |    0.6954 | EXT=0.560 NEU=0.405 AGR=0.517 CON=0.557 OPN=0.588 | AvgF1=0.526
    5 |      0.6603 |    0.7090 | EXT=0.555 NEU=0.492 AGR=0.506 CON=0.559 OPN=0.573 | AvgF1=0.537
    6 |      0.6499 |    0.6959 | EXT=0.515 NEU=0.398 AGR=0.504 CON=0.602 OPN=0.599 | AvgF1=0.524
    7 |      0.6275 |    0.7258 | EXT=0.491 NEU=0.421 AGR=0.497 CON=0.573 OPN=0.587 | AvgF1=0.514

Early stopping at epoch 7
Best validation F1: 0.5547
Model saved to: ../../model/LSTM\best_model_20260325-173143.pt

Test

In [8]:
# Compute combined classification report for all traits
gt_all   = {t: np.array(test_labels[t]) for t in TRAITS}
pred_all = {t: np.array(test_preds[t]) for t in TRAITS}

os.makedirs(EXP_DIR, exist_ok=True)

sep = "=" * 70
report_lines = [
    sep,
    "Personality Trait Detection — Classification Report",
    sep,
    f"Run ID      : {run_id}",
    f"Model       : PersonalityLSTM (BiLSTM)",
    f"Test file   : test.csv",
    f"Test samples: {len(test_df)}",
    f"Embed dim   : {EMBED_DIM}",
    f"Hidden dim  : {HIDDEN_DIM}",
    f"Num layers  : {NUM_LAYERS}",
    f"Max seq len : {MAX_LEN}",
    f"Epochs      : {len(history['epoch'])}",
    f"LR          : {LEARNING_RATE} | Batch size: {BATCH_SIZE}",
    f"Vocabulary  : {VOCAB_SIZE} (min_freq={MIN_FREQ})",
    sep,
]

for trait in TRAITS:
    gt   = gt_all[trait]
    pred = pred_all[trait]
    acc  = np.mean(gt == pred)
    f1   = f1_score(gt, pred, average="weighted", zero_division=0)

    report_str = classification_report(
        gt, pred,
        labels=[0, 1],
        target_names=["low", "high"],
        digits=4,
        zero_division=0,
    )

    report_lines.extend([
        f"\n--- {trait} ({TRAIT_NAMES[trait]}) ---",
        f"Accuracy: {acc:.4f} | F1 (weighted): {f1:.4f}",
        report_str,
    ])

report_lines.append(sep)

report_path = os.path.join(EXP_DIR, "classification_report.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))
print(f"Saved: {report_path}")

# Print per-trait summary to stdout
print(f"\n{'='*70}")
print(f"SUMMARY: Classification Results across OCEAN Traits")
print(f"{'='*70}")
print(f"{'Trait':<8} {'Test Acc':>10} {'Test F1':>10}")
print(f"{'-'*70}")

summary_data = []
for trait in TRAITS:
    gt   = gt_all[trait]
    pred = pred_all[trait]
    test_acc = np.mean(gt == pred)
    test_f1  = f1_score(gt, pred, average="weighted", zero_division=0)
    print(f"{trait:<8} {test_acc:>10.4f} {test_f1:>10.4f}")
    summary_data.append((trait, test_acc, test_f1))

print(f"{'-'*70}")
avg_test_acc = np.mean([s[1] for s in summary_data])
avg_test_f1  = np.mean([s[2] for s in summary_data])
print(f"{'AVERAGE':<8} {avg_test_acc:>10.4f} {avg_test_f1:>10.4f}")
print(f"{'='*70}")

Saved: ../../experiment/LSTM\classification_report.txt

SUMMARY: Classification Results across OCEAN Traits
Trait      Test Acc    Test F1
----------------------------------------------------------------------
cOPN         0.5749     0.5632
cCON         0.5668     0.5646
cEXT         0.5547     0.5448
cAGR         0.5992     0.5995
cNEU         0.5830     0.5811
----------------------------------------------------------------------
AVERAGE      0.5757     0.5707


In [9]:
# Save raw predictions with stable filename
test_pred_df = test_df.copy()
for trait in TRAITS:
    test_pred_df[f"{trait}_pred"] = test_preds[trait]
raw_pred_path = os.path.join(EXP_DIR, "raw_predictions.csv")
test_pred_df.to_csv(raw_pred_path, index=False)
print(f"Raw predictions saved: {raw_pred_path}")

Raw predictions saved: ../../experiment/LSTM\raw_predictions.csv


In [10]:
# Save vocabulary
vocab_path = os.path.join(MODEL_DIR, 'vocab.json')
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False)
print(f"Vocabulary saved: {vocab_path}")

# Save model weights package (for reloading)
model_weights_path = os.path.join(MODEL_DIR, 'model_weights.pt')
torch.save({
    'model_state_dict': torch.load(best_model_path, weights_only=True),
    'config': {
        'vocab_size': VOCAB_SIZE,
        'embed_dim': EMBED_DIM,
        'hidden_dim': HIDDEN_DIM,
        'num_layers': NUM_LAYERS,
        'dropout': DROPOUT,
        'num_traits': len(TRAITS),
        'max_len': MAX_LEN,
    },
    'traits': TRAITS,
    'run_id': run_id,
}, model_weights_path)
print(f"Model weights saved: {model_weights_path}")

# Save training config
config_path = os.path.join(MODEL_DIR, 'train_config.json')
train_config = {
    'max_len': MAX_LEN,
    'embed_dim': EMBED_DIM,
    'hidden_dim': HIDDEN_DIM,
    'num_layers': NUM_LAYERS,
    'dropout': DROPOUT,
    'batch_size': BATCH_SIZE,
    'epochs_trained': len(history['epoch']),
    'learning_rate': LEARNING_RATE,
    'early_stop_patience': EARLY_STOP_PATIENCE,
    'best_val_f1': float(best_val_f1),
    'test_loss': float(test_loss),
    'avg_test_acc': float(avg_test_acc),
    'avg_test_f1': float(avg_test_f1),
    'vocab_size': VOCAB_SIZE,
    'min_freq': MIN_FREQ,
    'run_id': run_id,
}
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(train_config, f, indent=2)
print(f"Training config saved: {config_path}")

print(f"\nAll artifacts saved.")
print(f"  Experiment: {EXP_DIR}/")
print(f"    - classification_report.txt")
print(f"    - raw_predictions.csv")
print(f"  Model: {MODEL_DIR}/")
print(f"    - best_model_{run_id}.pt")
print(f"    - model_weights.pt")
print(f"    - vocab.json")
print(f"    - train_config.json")

Vocabulary saved: ../../model/LSTM\vocab.json
Model weights saved: ../../model/LSTM\model_weights.pt
Training config saved: ../../model/LSTM\train_config.json

All artifacts saved.
  Experiment: ../../experiment/LSTM/
    - classification_report.txt
    - raw_predictions.csv
  Model: ../../model/LSTM/
    - best_model_20260325-173143.pt
    - model_weights.pt
    - vocab.json
    - train_config.json


In [11]:
# # Load saved model and run inference on new text
# import json

# # Reload model package
# LOAD_MODEL_DIR = '../model/LSTM'

# ckpt = torch.load(os.path.join(LOAD_MODEL_DIR, 'model_weights.pt'), weights_only=False)
# loaded_config = ckpt['config']
# loaded_vocab  = json.load(open(os.path.join(LOAD_MODEL_DIR, 'vocab.json'), encoding='utf-8'))

# print("Loaded model config:", loaded_config)
# print(f"Loaded vocabulary size: {len(loaded_vocab)}")

# # Instantiate model with saved config
# loaded_model = PersonalityLSTM(
#     vocab_size=loaded_config['vocab_size'],
#     embed_dim=loaded_config['embed_dim'],
#     hidden_dim=loaded_config['hidden_dim'],
#     num_layers=loaded_config['num_layers'],
#     dropout=loaded_config['dropout'],
#     num_traits=loaded_config['num_traits']
# )
# loaded_model.load_state_dict(ckpt['model_state_dict'])
# loaded_model = loaded_model.to(device)
# loaded_model.eval()
# print("Model loaded successfully.")

# # --- Inference helper ---
# def predict_text(text, model, vocab, max_len=MAX_LEN):
#     """
#     Given a single text string, return a dict of trait predictions (high/low).
#     """
#     encoded = encode(text, vocab, max_len)
#     x_tensor = torch.LongTensor([encoded]).to(device)

#     with torch.no_grad():
#         outputs = model(x_tensor)

#     predictions = {}
#     for trait in TRAITS:
#         prob = outputs[trait].item()
#         predictions[trait] = 'high' if prob >= 0.5 else 'low'
#         predictions[f'{trait}_prob'] = round(prob, 4)

#     return predictions

# # --- Quick sanity check on 2 test samples ---
# print("\n--- Sample predictions from loaded model ---")
# for i in range(2):
#     text = test_df.iloc[i]['text']
#     preds = predict_text(text, loaded_model, loaded_vocab)
#     print(f"\nSample {i+1}:")
#     for t in TRAITS:
#         gt  = test_df.iloc[i][t]
#         prob = preds[f'{t}_prob']
#         pred = preds[t]
#         match = "MATCH" if pred == gt else "MISMATCH"
#         print(f"  {t}: GT={gt}, pred={pred} (p={prob:.3f}) {match}")